## HILL CLIMBING

In [3]:
import string
import random
import numpy
import math

# Funzione per generare casualmente una stringa di lunghezza length
def generate_random_string( length = 8 ):
    return [random.choice(string.printable) for _ in range(length)]

# Funzione di valutazione, che valuta in base alla distanza tra le stringhe
def evaluate( solution ):

    # Stringa che vogliamo raggiungere
    target = list('Roma Tre')

    # Inizializzo le differenze
    diff = 0

    for i in range(len(solution)):

        s = solution[i]
        t = target[i]

        delta = abs( ord(s) - ord(t) )
        diff = diff + delta

    return diff

# Funzione che sceglie casualmente uno stato successore
def tweak( solution ):

    index = random.randint(0, len(solution) - 1)
    solution[index] = random.choice(string.printable)

def ricerca_stringa(stringa_iniziale):

    # Inizializzo lo stato corrente
    current_state = stringa_iniziale
    # Inizializzo la valutazione corrente
    current_eval = evaluate(current_state)

    while True:

        print('Punteggio attuale: ', current_eval, ' Stringa: ', "".join(current_state))

        # Se abbiamo raggiunto l'obiettivo
        if current_eval == 0:
            break

        # Genero lo stato successore
        next_state = list(current_state)
        tweak(next_state)
        next_state_eval = evaluate(next_state)

        # Se lo stato successore ha una valutazione migliore diventa lo stato corrente
        if next_state_eval < current_eval:
            current_state = next_state
            current_eval = next_state_eval

ricerca_stringa(generate_random_string())
    

Punteggio attuale:  293  Stringa:  +6bB/FM
Punteggio attuale:  293  Stringa:  +6bB/FM
Punteggio attuale:  249  Stringa:  +bbB/FM
Punteggio attuale:  249  Stringa:  +bbB/FM
Punteggio attuale:  249  Stringa:  +bbB/FM
Punteggio attuale:  249  Stringa:  +bbB/FM
Punteggio attuale:  249  Stringa:  +bbB/FM
Punteggio attuale:  249  Stringa:  +bbB/FM
Punteggio attuale:  249  Stringa:  +bbB/FM
Punteggio attuale:  222  Stringa:  +bbe/FM
Punteggio attuale:  213  Stringa:  +bbe/OM
Punteggio attuale:  213  Stringa:  +bbe/OM
Punteggio attuale:  213  Stringa:  +bbe/OM
Punteggio attuale:  213  Stringa:  +bbe/OM
Punteggio attuale:  213  Stringa:  +bbe/OM
Punteggio attuale:  213  Stringa:  +bbe/OM
Punteggio attuale:  213  Stringa:  +bbe/OM
Punteggio attuale:  208  Stringa:  +bge/OM
Punteggio attuale:  208  Stringa:  +bge/OM
Punteggio attuale:  208  Stringa:  +bge/OM
Punteggio attuale:  208  Stringa:  +bge/OM
Punteggio attuale:  208  Stringa:  +bge/OM
Punteggio attuale:  208  Stringa

## SIMULATED ANNEALING

In [ ]:
import random
import math
import numpy as np

# Impostazione parametri
TEMPERATURA_INIZIALE = 30
TEMPERATURA_FINALE = 0.2
ALFA = 0.99 # Coefficiente di diminuzione della temperatura
STEPS_PER_CHANGE = 100 # Numero di iterazioni per ogni temperatura
DIMENSIONE = 8 # Dimensione N x N della scacchiera dove N è DIMENSIONE

def tweak(sol): # Sceglie randomicamente uno stato successore

    # Faccio una copia dello stato attuale
    sol_copy = np.copy(sol)

    # Trovo due indici casuali
    x = random.randint(0, DIMENSIONE - 1)
    y = random.randint(0, DIMENSIONE - 1)

    while x==y:
        y= random.randint(0, DIMENSIONE - 1)

    # Scambio i due indici
    temp = sol_copy[y]
    sol_copy[y] = sol_copy[x]
    sol_copy[x] = temp

    return sol_copy

def inizializza(sol): # Sceglie random uno stato iniziale

    for c in range(0, DIMENSIONE-1):
        sol = tweak(sol)

    return sol

def energia(stato): # Funzione di valutazione basata, sui conflitti tra regine ( regine che attaccano altre regine )

    # Inizializzo la scacchiera
    board = [ [0]*DIMENSIONE for i in range(DIMENSIONE)]

    # Riempio la scacchiera secondo l'encoding 
    for i in range(0, DIMENSIONE):
        board[stato[i]][i] = 'Q'

    # Possibili spostamenti
    dx = [1, -1, 1, -1]
    dy = [1, -1, -1, 1]

    # Inizializzo i conflitti
    conflitti = 0

    for i in range(0, DIMENSIONE): # Per ogni regina trovo i conflitti
        x = stato[i]
        y = i
        for j in range(0, 4): # Per tutte le direzioni possibili
            tempx = x
            tempy = y
            while True: # Lungo tutta la diagonale
                tempx = tempx + dx[j]
                tempy = tempy + dy[j]
                if tempx < 0 or tempx>=DIMENSIONE or tempy<0 or tempy>=DIMENSIONE: # Se esco dalla scacchiera mi fermo
                    break
                if board[tempx][tempy]=='Q': # Ho trovato una regina segno il conflitto
                    conflitti = conflitti + 1

    return conflitti

def simulated_annealing():

    # Inizializzo lo stato corrente
    current = inizializza(range(0, DIMENSIONE))
    # Inizializzo l'energia corrente
    current_energy = energia(current)

    # Inizializzo il best
    best = current
    best_energy = current_energy

    # Inizializzo la temperatura
    temperature = TEMPERATURA_INIZIALE

    # Fintanto che non ho raggiunto la temperatura finale o ho trovato l'obiettivo
    while temperature > TEMPERATURA_FINALE and best_energy != 0:

        print('Temperatura: ', temperature)

        for step in range(0, STEPS_PER_CHANGE): # Per ogni temperatura itero un certo numero (STEPS_PER_CHANGE) di volte
            useNew = False # variabile che mi dice se accettare il next state
            next_state = tweak(current)
            next_state_energy = energia(next_state)

            if next_state_energy < current_energy: # Se il next state è migliore lo considero
                useNew = True
            else: # Sennò calcolo la probabilità di accettazione
                delta = next_state_energy - current_energy # Calcolo di quanto è peggiore il next state
                metropolis = math.exp(-delta/temperature)
                test = random.random() # Estraggo un numero casuale tra 0 e 1
                if test < metropolis: # Se la probabilità di accettazione è maggiore del numero casuale accetto il next_state
                    useNew = True

            if(useNew):
                current = next_state
                current_energy = next_state_energy

                if current_energy < best_energy:
                    best = current
                    best_energy = current_energy
            
        print('Best energy: ', best_energy)
        temperature = temperature * ALFA # Diminuisco la temperatura

    return best

simulated_annealing()

TEMPERATURA: 
30
best_energy=
2
TEMPERATURA: 
29.7
best_energy=
2
TEMPERATURA: 
29.403
best_energy=
2
TEMPERATURA: 
29.10897
best_energy=
2
TEMPERATURA: 
28.8178803
best_energy=
0


array([0, 4, 7, 5, 2, 6, 1, 3])

## A STAR TREE SEARCH

In [ ]:
class Node: # Per i nodi dell'albero di ricerca

    def __init__(self, state, parent, f, partial_path):

        self.state = state
        self.parent = parent
        self.heuristc = f
        self.partial_path = partial_path

    def printPath(self): # Metodo che stampa ricorsivamente il cammino dal nodo iniziale al nodo corrente

        if self.parent!=None:
            self.parent.printPath()
        print('->', self.state.name)
    
class State: # Per gli stati dello spazio degli stati

    def __init__(self, name = None):

        if self.name == None: # Creo lo stato iniziale
            self.name = self.getInitialState()
        else:
            self.name = name

    def getInitialState(self):
        initialState = 'Arad'
        return initialState
    
    def successorFunction(self):
        return connections[self.name]
    
    def checkGoalState(self):
        return self.name == 'Bucarest'
    
class Elem: # Per gli elementi della frontiera

    val = None
    node = None
    next = None

    def __init__(self, val, node):
        self.val = val
        self.node = node
        self.next = None

class Fringe: # Classe per la frontiera

    def __init__(self):
        self.__head = None
        self.__tail = None

    def add(self, newNode):

        if self.__head == None: # Inserimento in frontiera nuova
            self.__head = newNode
            self.__head = self.__tail
            newNode.next = None
        elif newNode.val > self.__tail.val: # Inserimento in coda (append)
            self.__tail.next = newNode
            self.__tail = newNode
            newNode.next = None
        elif newNode.val < self.__head.val: # Inserimento in testa
            newNode.next = self.__head
            self.__head = newNode
        else:
            p= self.__head
            while p.next!=None and newNode.val > p.next.val :
                p = p.next
            newNode.next = p.next
            p.next = newNode
    
    # Estrae il primo elemento della frontiera
    def estrazione(self):

        if self.__head == None:
            return None
        else:
            p = self.__head
            self.__head = self.__head.next
            p.next = None
            return p
        
    # Restituisce True se la frontiera è vuota
    def empty_fringe(self):

        if self.__head == None:
            return True
        else:
            return False
    
    def stampa(self):

        print('Head', end = ' ')
        p = self.__head
        while p!=None:
            print(p.node.state.name, '->', end = ' ')
            p = p.next
        print('Tail')

def A_Star():

    # Creo la frontiera
    fringe = Fringe()

    # Creo lo stato iniziale
    initialState = State()

    # Creo la radice
    euristica = h[initialState.name]
    root = Node(initialState, None, euristica, 0)

    # Aggiungo in frontiera
    elem = Elem(euristica, root)
    fringe.add(elem)

    # Fintanto che la frontiera non è vuota
    while not fringe.empty_fringe():

        elem_estratto = fringe.estrazione()
        currentNode = elem_estratto.node

        print('--Dequeue--', currentNode.state.name)

        if currentNode.state.checkGoalState():

            print('Stato obiettivo raggiunto')
            print('---')
            print('Soluzione: ')
            currentNode.printPath()
            break
        else:

            childStates = currentNode.state.successorFunction()

            for (childState, distance) in childStates:

                # Creo il nodo figlio
                euristica = h[State(childState).name]
                g = currentNode.partial_path + distance
                f = g + euristica
                currentChildNode = Node(State(childState), currentNode, f, g)

                # Aggiungo il nodo figlio in frontiera
                elemento = Elem(currentChildNode.heuristc, currentChildNode)
                fringe.add(elemento)



## A STAR GRAPH SEARCH

In [ ]:
class Node: # Per i nodi dell'albero di ricerca

    def __init__(self, state, parent, f, partial_path):

        self.state = state
        self.parent = parent
        self.heuristc = f
        self.partial_path = partial_path

    def printPath(self): # Metodo che stampa ricorsivamente il cammino dal nodo iniziale al nodo corrente

        if self.parent!=None:
            self.parent.printPath()
        print('->', self.state.name)
    
class State: # Per gli stati dello spazio degli stati

    def __init__(self, name = None):

        if self.name == None: # Creo lo stato iniziale
            self.name = self.getInitialState()
        else:
            self.name = name

    def getInitialState(self):
        initialState = 'Arad'
        return initialState
    
    def successorFunction(self):
        return connections[self.name]
    
    def checkGoalState(self):
        return self.name == 'Bucarest'
    
class Elem: # Per gli elementi della frontiera

    val = None
    node = None
    next = None

    def __init__(self, val, node):
        self.val = val
        self.node = node
        self.next = None

class Fringe: # Classe per la frontiera

    def __init__(self):
        self.__head = None
        self.__tail = None

    def add(self, newNode):

        if self.__head == None: # Inserimento in frontiera nuova
            self.__head = newNode
            self.__head = self.__tail
            newNode.next = None
        elif newNode.val > self.__tail.val: # Inserimento in coda (append)
            self.__tail.next = newNode
            self.__tail = newNode
            newNode.next = None
        elif newNode.val < self.__head.val: # Inserimento in testa
            newNode.next = self.__head
            self.__head = newNode
        else:
            p= self.__head
            while p.next!=None and newNode.val > p.next.val :
                p = p.next
            newNode.next = p.next
            p.next = newNode
    
    # Estrae il primo elemento della frontiera
    def estrazione(self):

        if self.__head == None:
            return None
        else:
            p = self.__head
            self.__head = self.__head.next
            p.next = None
            return p
        
    # Restituisce True se la frontiera è vuota
    def empty_fringe(self):

        if self.__head == None:
            return True
        else:
            return False
    
    def stampa(self):

        print('Head', end = ' ')
        p = self.__head
        while p!=None:
            print(p.node.state.name, '->', end = ' ')
            p = p.next
        print('Tail')

def A_Star():

    # Creo la frontiera
    fringe = Fringe()

    # Creo la lista di nodi espansi
    close = []

    # Creo lo stato iniziale
    initialState = State()

    # Creo la radice
    euristica = h[initialState.name]
    root = Node(initialState, None, euristica, 0)

    # Aggiungo in frontiera
    elem = Elem(euristica, root)
    fringe.add(elem)

    # Fintanto che la frontiera non è vuota
    while not fringe.empty_fringe():

        elem_estratto = fringe.estrazione()
        currentNode = elem_estratto.node

        print('--Dequeue--', currentNode.state.name)

        if currentNode.state.checkGoalState():

            print('Stato obiettivo raggiunto')
            print('---')
            print('Soluzione: ')
            currentNode.printPath()
            break
        else:

            if currentNode.state.name not in close:

                close.append(currentNode.state.name)

                childStates = currentNode.state.successorFunction()

                for (childState, distance) in childStates:

                    # Creo il nodo figlio
                    euristica = h[State(childState).name]
                    g = currentNode.partial_path + distance
                    f = g + euristica
                    currentChildNode = Node(State(childState), currentNode, f, g)

                    if currentChildNode.state.name not in close:

                        # Aggiungo il nodo figlio in frontiera
                        elemento = Elem(currentChildNode.heuristc, currentChildNode)
                        fringe.add(elemento)



## GREEDY

In [ ]:
class Node: # Per i nodi dell'albero di ricerca

    def __init__(self, state, parent, h):

        self.state = state
        self.parent = parent
        self.heuristc = h

    def printPath(self): # Metodo che stampa ricorsivamente il cammino dal nodo iniziale al nodo corrente

        if self.parent!=None:
            self.parent.printPath()
        print('->', self.state.name)
    
class State: # Per gli stati dello spazio degli stati

    def __init__(self, name = None):

        if self.name == None: # Creo lo stato iniziale
            self.name = self.getInitialState()
        else:
            self.name = name

    def getInitialState(self):
        initialState = 'Arad'
        return initialState
    
    def successorFunction(self):
        return connections[self.name]
    
    def checkGoalState(self):
        return self.name == 'Bucarest'
    
class Elem: # Per gli elementi della frontiera

    val = None
    node = None
    next = None

    def __init__(self, val, node):
        self.val = val
        self.node = node
        self.next = None

class Fringe: # Classe per la frontiera

    def __init__(self):
        self.__head = None
        self.__tail = None

    def add(self, newNode):

        if self.__head == None: # Inserimento in frontiera nuova
            self.__head = newNode
            self.__head = self.__tail
            newNode.next = None
        elif newNode.val > self.__tail.val: # Inserimento in coda (append)
            self.__tail.next = newNode
            self.__tail = newNode
            newNode.next = None
        elif newNode.val < self.__head.val: # Inserimento in testa
            newNode.next = self.__head
            self.__head = newNode
        else:
            p= self.__head
            while p.next!=None and newNode.val > p.next.val :
                p = p.next
            newNode.next = p.next
            p.next = newNode
    
    # Estrae il primo elemento della frontiera
    def estrazione(self):

        if self.__head == None:
            return None
        else:
            p = self.__head
            self.__head = self.__head.next
            p.next = None
            return p
        
    # Restituisce True se la frontiera è vuota
    def empty_fringe(self):

        if self.__head == None:
            return True
        else:
            return False
    
    def stampa(self):

        print('Head', end = ' ')
        p = self.__head
        while p!=None:
            print(p.node.state.name, '->', end = ' ')
            p = p.next
        print('Tail')

def greedy_best_first_search():

    # Creo la frontiera
    fringe = Fringe()

    # Creo lo stato iniziale
    initialState = State()

    # Creo la radice
    euristica = h[initialState.name]
    root = Node(initialState, None, euristica)

    # Aggiungo in frontiera
    elem = Elem(euristica, root)
    fringe.add(elem)

    # Fintanto che la frontiera non è vuota
    while not fringe.empty_fringe():

        elem_estratto = fringe.estrazione()
        currentNode = elem_estratto.node

        print('--Dequeue--', currentNode.state.name)

        if currentNode.state.checkGoalState():

            print('Stato obiettivo raggiunto')
            print('---')
            print('Soluzione: ')
            currentNode.printPath()
            break
        else:

            childStates = currentNode.state.successorFunction()

            for childState in childStates:

                # Creo il nodo figlio
                euristica = h[State(childState).name]
                currentChildNode = Node(State(childState), currentNode, euristica)

                # Aggiungo il nodo figlio in frontiera
                elemento = Elem(currentChildNode.heuristc, currentChildNode)
                fringe.add(elemento)

